# 04 — Modelo 1: Detección Binaria de Anomalías

Entrenamos un Random Forest para clasificar cada fila como `normal` o `anomalia`.

Mejoras v3 respecto a v2:
- **M6**: TimeSeriesSplit con k=4 en lugar de split aleatorio 70/30 → sin data leakage temporal
- Features enriquecidas con M1 + M2

**Entrada:** `data/interim/03_datos_features.parquet`  
**Salida:** modelos en `data/models/` + `data/interim/04_modelo1_predicciones.parquet`

In [ ]:
# ─── Intel Extension for Scikit-learn ────────────────────────────────────────
# IMPORTANTE: debe cargarse ANTES que cualquier import de sklearn.
# Instalar con: pip install scikit-learn-intelex
# En CPU Intel puede acelerar Random Forest e IterativeImputer 10-100x.
try:
    from sklearnex import patch_sklearn
    patch_sklearn()
    _intel_activo = True
except ImportError:
    _intel_activo = False

# ─── Librerías estándar ───────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import os, glob, joblib, pickle, warnings
import matplotlib.ticker as mticker

# ─── Scikit-learn (ya parcheado si Intel Extension está disponible) ───────────
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.experimental import enable_iterative_imputer  # requerido antes de IterativeImputer
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              classification_report, confusion_matrix,
                              average_precision_score)
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100

# ─── Configuración del proyecto ───────────────────────────────────────────────
import sys
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from config import *

if _intel_activo:
    print("Intel Extension for Scikit-learn ACTIVA — algoritmos acelerados con oneAPI")
else:
    print("Tip: instala scikit-learn-intelex para acelerar en CPU Intel")
    print("     pip install scikit-learn-intelex")


## 0. Cargar datos y preparar X, y

In [ ]:
# Cargar el dataset con features calculadas
df_trabajo = pd.read_parquet(PARQUET_03)
print(f"Dataset cargado: {df_trabajo.shape}")

# Separar features (X) y etiqueta binaria (y)
excluir = COLUMNAS_EXCLUIR_FEATURES
columnas_features = [c for c in df_trabajo.columns if c not in excluir]
X = df_trabajo[columnas_features].select_dtypes(include='number').copy()
y = df_trabajo['etiqueta_deteccion'].copy()

print(f"Features: {X.shape[1]} columnas")
print(f"Distribución de clases:")
print(y.value_counts())


## 1. TimeSeriesSplit — k=4 folds (v3 — M6)
Dividimos en 4 partes ordenadas temporalmente. Esto evita data leakage: el modelo nunca ve datos futuros durante el entrenamiento.
Calculamos accuracy, F1 y ROC-AUC por fold para ver estabilidad temporal.

In [ ]:
# v3 — M6: TimeSeriesSplit con k=4 folds — sin data leakage, evaluación en todas las estaciones
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

if 'X' in locals() and 'y' in locals():
    tscv = TimeSeriesSplit(n_splits=4)

    resultados_folds = []
    fold_splits = []  # guardamos los índices para reusar el último fold en corrección

    print('TimeSeriesSplit k=4 — distribución de folds:')
    print('=' * 60)

    for fold, (train_idx, test_idx) in enumerate(tscv.split(X), start=1):
        X_train_f = X.iloc[train_idx]
        X_test_f  = X.iloc[test_idx]
        y_train_f = y.iloc[train_idx]
        y_test_f  = y.iloc[test_idx]

        # Fechas aproximadas de cada fold (si Fecha está disponible)
        if 'Fecha' in df_trabajo.columns:
            fecha_train_ini = df_trabajo['Fecha'].iloc[train_idx[0]]
            fecha_train_fin = df_trabajo['Fecha'].iloc[train_idx[-1]]
            fecha_test_ini  = df_trabajo['Fecha'].iloc[test_idx[0]]
            fecha_test_fin  = df_trabajo['Fecha'].iloc[test_idx[-1]]
            print(f'Fold {fold}:')
            print(f'  Train: {fecha_train_ini:%Y-%m-%d} → {fecha_train_fin:%Y-%m-%d} ({len(train_idx):,} filas)')
            print(f'  Test:  {fecha_test_ini:%Y-%m-%d} → {fecha_test_fin:%Y-%m-%d} ({len(test_idx):,} filas)')
        else:
            print(f'Fold {fold}: train={len(train_idx):,} filas  test={len(test_idx):,} filas')

        fold_splits.append((train_idx, test_idx))

    print('=' * 60)
    print()
    print('Los 4 modelos se entrenan en la celda siguiente.')
    print('El Fold 4 (train más largo) se usa para corrección y comparación final.')

    # Exponer el fold 4 como split principal para el resto del notebook
    train_idx_final, test_idx_final = fold_splits[-1]
    X_train = X.iloc[train_idx_final]
    X_test  = X.iloc[test_idx_final]
    y_train = y.iloc[train_idx_final]
    y_test  = y.iloc[test_idx_final]

else:
    print('Error: X o y no definidos.')


## 2. Entrenar modelo final en el último fold
Usamos el split del fold final (más reciente) para tener el modelo entrenado con el mayor volumen de datos posible.

In [ ]:
# v3 — M6: Entrenar un modelo por fold y calcular métricas por estación
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, classification_report

if 'fold_splits' in locals() and 'X' in locals() and 'y' in locals():
    metricas_folds = []

    for fold, (train_idx, test_idx) in enumerate(fold_splits, start=1):
        X_tr = X.iloc[train_idx]
        X_te = X.iloc[test_idx]
        y_tr = y.iloc[train_idx]
        y_te = y.iloc[test_idx]

        # Imputar NaN por columna
        imp = SimpleImputer(strategy='median', add_indicator=True)
        X_tr_imp = imp.fit_transform(X_tr)
        X_te_imp = imp.transform(X_te)

        # Entrenar RF
        rf = RandomForestClassifier(n_estimators=100, random_state=42,
                                     class_weight='balanced', n_jobs=-1)
        rf.fit(X_tr_imp, y_tr)
        y_pred = rf.predict(X_te_imp)

        acc = accuracy_score(y_te, y_pred)
        f1  = f1_score(y_te, y_pred, average='weighted')

        if 'Fecha' in df_trabajo.columns:
            fecha_test_ini = df_trabajo['Fecha'].iloc[test_idx[0]]
            fecha_test_fin = df_trabajo['Fecha'].iloc[test_idx[-1]]
            periodo = f'{fecha_test_ini:%b %Y} → {fecha_test_fin:%b %Y}'
        else:
            periodo = f'filas {test_idx[0]}–{test_idx[-1]}'

        metricas_folds.append({
            'Fold': fold,
            'Período test': periodo,
            'Accuracy': round(acc * 100, 2),
            'F1-weighted': round(f1, 4),
            'Train filas': len(train_idx),
            'Test filas': len(test_idx),
        })

        print(f'Fold {fold} ({periodo}): Accuracy={acc*100:.2f}%  F1={f1:.4f}')

        # El modelo del fold 4 es el principal para el resto del notebook
        if fold == len(fold_splits):
            rf_detector      = rf
            imputer          = imp
            X_train_imputed  = X_tr_imp
            X_test_imputed   = X_te_imp
            y_pred_detector  = y_pred

    import pandas as pd
    df_metricas = pd.DataFrame(metricas_folds)
    print()
    print('=' * 65)
    print('RESUMEN TimeSeriesSplit k=4')
    print('=' * 65)
    print(df_metricas.to_string(index=False))
    print(f"\nMedia  Accuracy: {df_metricas['Accuracy'].mean():.2f}% ± {df_metricas['Accuracy'].std():.2f}%")
    print(f"Media  F1:       {df_metricas['F1-weighted'].mean():.4f} ± {df_metricas['F1-weighted'].std():.4f}")
    print('=' * 65)
    print('Modelo del Fold 4 guardado como rf_detector para corrección.')

else:
    print('Error: fold_splits no definido. Ejecuta la celda anterior.')


## 3. Imputar NaNs con SimpleImputer + indicadores
El imputer rellena NaNs con la mediana y añade columnas binarias `missingindicator_X` para que el modelo sepa dónde había NaN (el hecho de ser NaN es en sí una señal de anomalía).

In [ ]:
if 'X_train' in locals() and 'X_test' in locals():
    print("Manejo de NaNs antes de la imputación:")
    print(f"NaNs en X_train: {X_train.isnull().sum().sum()}")
    print(f"NaNs en X_test: {X_test.isnull().sum().sum()}")

    # Crear el imputador usando la mediana Y AÑADIENDO COLUMNAS INDICADORAS
    imputer = SimpleImputer(strategy='median', add_indicator=True)

    # Ajustar el imputador SOLO con los datos de entrenamiento
    # Nota: fit no devuelve nada, solo configura el imputer
    imputer.fit(X_train) 
    
    # Aplicar la imputación (transformar) a los datos de entrenamiento y prueba
    # El resultado será un array de NumPy
    X_train_imputed_np = imputer.transform(X_train)
    X_test_imputed_np = imputer.transform(X_test)

    # Obtener los nombres de las nuevas características (originales + indicadoras)
    try:
        # Intenta obtener los nombres de las características, incluyendo las indicadoras
        feature_names_imputed = imputer.get_feature_names_out(input_features=X_train.columns)
    except AttributeError:
        print("Advertencia: imputer.get_feature_names_out no disponible. Usando nombres genéricos para indicadoras si es necesario.") 
        # Si no se puede obtener, usamos un enfoque alternativo
        feature_names_imputed = list(X_train.columns)
        # Ver cuántas columnas indicadoras se añadieron
        num_original_features = X_train.shape[1]
        num_indicator_features = X_train_imputed_np.shape[1] - num_original_features
        if num_indicator_features > 0:
            # Obtener índices de características que tenían NaNs y para las cuales se creó un indicador
            indicator_indices = imputer.indicator_.features_ # Indices de las cols originales que generaron indicador
            indicator_col_names = [f"missing_{X_train.columns[i]}" for i in indicator_indices]
            feature_names_imputed.extend(indicator_col_names)
        if len(feature_names_imputed) != X_train_imputed_np.shape[1]: # Comprobación de seguridad
            print(f"Discrepancia en nombres de columnas: {len(feature_names_imputed)} vs {X_train_imputed_np.shape[1]}. Feature importance podría no tener nombres correctos.")
            feature_names_imputed = None # Forzar a usar array si los nombres no cuadran


    # Convertir de nuevo a DataFrames de Pandas
    if feature_names_imputed is not None:
        X_train_imputed = pd.DataFrame(X_train_imputed_np, columns=feature_names_imputed, index=X_train.index)
        X_test_imputed = pd.DataFrame(X_test_imputed_np, columns=feature_names_imputed, index=X_test.index)
    else: # Si no pudimos obtener los nombres, trabajamos con arrays de NumPy (menos ideal para feature importance explícita)
        X_train_imputed = X_train_imputed_np
        X_test_imputed = X_test_imputed_np
        print("Advertencia: X_train_imputed y X_test_imputed son arrays NumPy. Nombres de características no disponibles para la tabla de importancia.")


    print("\nManejo de NaNs después de la imputación (con indicadores):")
    # Si son DataFrames, podemos hacer isnull().sum().sum(). Si son arrays, no directamente.
    if isinstance(X_train_imputed, pd.DataFrame):
        print(f"NaNs en X_train_imputed: {X_train_imputed.isnull().sum().sum()}")
        print(f"NaNs en X_test_imputed: {X_test_imputed.isnull().sum().sum()}")
    else: # Si son arrays NumPy, SimpleImputer debería haber manejado todos los NaNs
        print(f"NaNs en X_train_imputed (array NumPy): {np.isnan(X_train_imputed).sum()}")
        print(f"NaNs en X_test_imputed (array NumPy): {np.isnan(X_test_imputed).sum()}")

    print(f"Forma de X_train_imputed: {X_train_imputed.shape} (puede tener más columnas debido a los indicadores)")
    print(f"Forma de X_test_imputed: {X_test_imputed.shape}")
else:
    print("Error: X_train o X_test no están definidos. Ejecuta los pasos anteriores.")

## 4. Entrenar Random Forest Detector

In [ ]:
if 'X_train_imputed' in locals() and 'y_train' in locals():
    # Inicializar el clasificador Random Forest
    # class_weight='balanced' puede ayudar si hay desbalance entre clases 'normal' y 'anomalia'
    # n_estimators: número de árboles en el bosque.
    # random_state: para reproducibilidad.
    rf_detector = RandomForestClassifier(n_estimators=100,
                                            random_state=42,
                                            class_weight='balanced', # Útil si las clases están desbalanceadas
                                            n_jobs=-1) # Usar todos los procesadores disponibles

    print("\nEntrenando el modelo Random Forest Detector...")
    rf_detector.fit(X_train_imputed, y_train)
    print("Entrenamiento completado.")

else:
    print("Error: X_train_imputed o y_train no están definidos.")

## 5. Evaluar en test: métricas globales

In [ ]:
if 'rf_detector' in locals() and 'X_test_imputed' in locals() and 'y_test' in locals():
    print("\nRealizando predicciones en el conjunto de prueba...")
    y_pred_detector = rf_detector.predict(X_test_imputed)

    # Orden semántico correcto: Normal=negativo, Anomalía=positivo
    labels_orden  = ['normal', 'anomalia']
    display_names = ['Normal', 'Anomalía']
    idx_anomalia  = list(rf_detector.classes_).index('anomalia')
    y_pred_proba_anomalia = rf_detector.predict_proba(X_test_imputed)[:, idx_anomalia]

    print("\n--- Evaluación del Modelo Detector ---")
    accuracy = accuracy_score(y_test, y_pred_detector)
    print(f"Accuracy: {accuracy:.4f}")

    # Matriz de Confusión con orden correcto
    print("\nMatriz de Confusión:")
    cm = confusion_matrix(y_test, y_pred_detector, labels=labels_orden)
    print(cm)

    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=display_names, yticklabels=display_names)
    plt.xlabel('Predicción')
    plt.ylabel('Valor Real')
    plt.title('Matriz de Confusión - Detector de Anomalías')
    plt.tight_layout()
    plt.show()

    # TN/FP/FN/TP con 'anomalia' como clase positiva
    tn, fp, fn, tp = cm.ravel()
    print(f"\nAnálisis Adicional del Detector:")
    print(f"  - Verdaderos Positivos (VP - Anomalías detectadas):        {tp}")
    print(f"  - Falsos Negativos (FN - Anomalías NO detectadas):         {fn}")
    print(f"  - Falsos Positivos (FP - Normales marcados como anomalía): {fp}")
    print(f"  - Verdaderos Negativos (TN - Normales correctamente):      {tn}")

    # Reporte de Clasificación
    print("\nReporte de Clasificación:")
    print(classification_report(y_test, y_pred_detector,
                                labels=labels_orden, target_names=display_names))

    # ROC AUC y PR AUC usando probabilidad de 'anomalia'
    y_bin = (y_test == 'anomalia').astype(int)
    roc_auc = roc_auc_score(y_bin, y_pred_proba_anomalia)
    pr_auc  = average_precision_score(y_bin, y_pred_proba_anomalia)
    print(f"ROC AUC Score:                         {roc_auc:.4f}")
    print(f"Precision-Recall AUC (Average Precision): {pr_auc:.4f}")

else:
    print("Error: El modelo 'rf_detector' no está entrenado o los datos de prueba no están disponibles.")

## 6. Importancia de features
¿Qué variables son más relevantes para detectar anomalías?

In [ ]:
if ('rf_detector' in locals() and
    hasattr(rf_detector, 'feature_importances_') and
    'X_train_imputed' in locals()): # Verificar X_train_imputed

    if not isinstance(X_train_imputed, pd.DataFrame):
        print("Error: X_train_imputed no es un DataFrame de Pandas. No se pueden obtener nombres de características.")
        print("Esto puede suceder si la construcción de feature_names_imputed falló en el paso de imputación.")
    else:
        importances = rf_detector.feature_importances_
        feature_names = X_train_imputed.columns
        
        # Verificar la longitud para evitar el error
        if len(feature_names) == len(importances):
            # Crear un DataFrame para visualizar mejor
            feature_importance_df = pd.DataFrame({'Característica': feature_names, 'Importancia': importances})
            feature_importance_df = feature_importance_df.sort_values(by='Importancia', ascending=False)
            
            print("\n--- Importancia de las Características (Detector de Anomalías) ---")
            print(feature_importance_df.to_string())
            
            # Graficar la importancia de las características
            plt.figure(figsize=(12, max(6, len(feature_names) * 0.4))) 
            plt.barh(feature_importance_df['Característica'], feature_importance_df['Importancia'], color='mediumpurple')
            plt.xlabel("Importancia Relativa")
            plt.ylabel("Característica")
            plt.title("Importancia de las Características - Detector de Anomalías (Random Forest)")
            plt.gca().invert_yaxis() 
            plt.tight_layout() 
            plt.show()
        else:
            print(f"Error de coincidencia de longitud: Nombres de características ({len(feature_names)}) vs. Importancias ({len(importances)}).")
            print("Asegúrate de que X_train_imputed tenga los nombres de columna correctos después de la imputación.")

else:
    print("Error: No se pudo calcular la importancia de las características.")
    print("Asegúrate de que 'rf_detector' esté entrenado y 'X_train_imputed' (como DataFrame con columnas) esté disponible.")

## 7. Desglose por tipo de anomalía
¿Cuántas anomalías de cada tipo detecta correctamente el modelo?

In [ ]:
if ('df_trabajo' in locals() and
    'X_test' in locals() and
    'y_test' in locals() and
    'y_pred_detector' in locals()):

    # Obtener los tipos de anomalía reales para el conjunto de prueba
    # X_test.index contiene los índices originales de df_trabajo para las filas de prueba
    true_anomaly_types_in_test_set = df_trabajo.loc[X_test.index, 'etiqueta_tipo_anomalia']
    
    # Crear el DataFrame para análisis
    df_eval_detector = pd.DataFrame({
        'indice_original': X_test.index,
        'y_real_binaria': y_test,          # 'normal' o 'anomalia'
        'y_pred_detector': y_pred_detector, # 'normal' o 'anomalia' predicho
        'tipo_anomalia_real': true_anomaly_types_in_test_set
    })

    print("\n--- Desglose de Detección del Modelo 1 por Tipo Real de Anomalía (en Conjunto de Prueba) ---")

    # Filtrar solo las filas que SON REALMENTE ANOMALÍAS
    df_anomalias_reales_en_test = df_eval_detector[df_eval_detector['y_real_binaria'] == 'anomalia']

    if df_anomalias_reales_en_test.empty:
        print("No hay anomalías reales en el conjunto de prueba para analizar.")
    else:
        # Agrupar por el tipo real de anomalía y contar cuántas fueron detectadas
        detection_summary = df_anomalias_reales_en_test.groupby('tipo_anomalia_real').agg(
            total_instancias_reales_en_test = pd.NamedAgg(column='y_pred_detector', aggfunc='count'),
            correctamente_detectadas_como_anomalia = pd.NamedAgg(column='y_pred_detector', aggfunc=lambda x: (x == 'anomalia').sum()),
            no_detectadas_por_el_detector_FN = pd.NamedAgg(column='y_pred_detector', aggfunc=lambda x: (x == 'normal').sum())
        ).reset_index()

        # Calcular tasa de detección por tipo
        detection_summary['tasa_deteccion_%'] = (
            detection_summary['correctamente_detectadas_como_anomalia'] / 
            detection_summary['total_instancias_reales_en_test'] * 100
        )
        
        # Ordenar por tasa de detección o total de instancias para mejor visualización
        detection_summary = detection_summary.sort_values(by='tasa_deteccion_%', ascending=False)

        print("\nResumen de detección por tipo de anomalía real:")
        print(detection_summary.to_string())
        
        # Graficar la tasa de detección por tipo de anomalía
        if not detection_summary.empty:
            plt.figure(figsize=(12, max(6, len(detection_summary) * 0.5)))
            
            detection_summary.set_index('tipo_anomalia_real')[['correctamente_detectadas_como_anomalia', 'no_detectadas_por_el_detector_FN']].plot(
                kind='barh', 
                stacked=True, 
                figsize=(12, max(6, len(detection_summary) * 0.6)),
                colormap='viridis'
            )
            plt.title('Rendimiento del Detector por Tipo de Anomalía Real (en Conjunto de Prueba)')
            plt.xlabel('Número de Instancias')
            plt.ylabel('Tipo de Anomalía Real')
            plt.legend(title='Resultado Detección', loc='lower right')
            plt.tight_layout()
            plt.show()

            plt.figure(figsize=(10, max(5, len(detection_summary) * 0.5) ))
            sns.barplot(x='tasa_deteccion_%', y='tipo_anomalia_real', data=detection_summary, palette='magma', hue='tipo_anomalia_real', dodge=False, legend=False)
            plt.title('Tasa de Detección del Modelo 1 por Tipo de Anomalía Real (%)')
            plt.xlabel('Tasa de Detección (%)')
            plt.ylabel('Tipo de Anomalía Real')
            plt.xlim(0, 105)
            for index, row in detection_summary.iterrows():
                plt.text(row['tasa_deteccion_%'] + 1,
                        index,
                        f"{row['tasa_deteccion_%']:.1f}%", 
                        color='black', 
                        ha="left", 
                        va="center")
            plt.tight_layout()
            plt.show()

else:
    print("Error: No se pueden generar los datos de desglose.")
    print("Asegúrate de que 'df_trabajo', 'X_test', 'y_test', y 'y_pred_detector' estén disponibles.")

## 8. Guardar modelo y resultados

In [ ]:
# Guardar el modelo entrenado, el imputer y la lista de features
os.makedirs(DATA_MODELS, exist_ok=True)
joblib.dump(rf_detector,     MODELO_1_PATH)
joblib.dump(imputer,         IMPUTER_M1_PATH)
joblib.dump(list(X_train_imputed.columns), FEATURES_M1_PATH)
print(f"Modelo 1 guardado en: {MODELO_1_PATH}")

# Guardar predicciones sobre todo el dataset para usarlas en el Modelo 2
df_trabajo['pred_deteccion'] = rf_detector.predict(
    pd.DataFrame(imputer.transform(X), columns=X_train_imputed.columns)
)
df_trabajo.to_parquet(PARQUET_04, index=False)
print(f"Predicciones guardadas: {PARQUET_04}")
